# 02 — Modeling and controlled model selection

The full benchmark is generated by `scripts/run_benchmark.py`. This notebook inspects the resulting evidence and independently fits the selected logistic pipeline as a smoke test.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import matplotlib.pyplot as plt
from telco_churn.data import load_splits
from telco_churn.features import add_active_product_count, split_xy
frames = load_splits(ROOT / 'data' / 'synthetic')

from telco_churn.modeling import build_candidate_models, predict_scores
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
validation_results = pd.read_csv(ROOT / 'reports' / 'metrics' / 'validation_model_comparison.csv')
validation_results[['model','roc_auc','pr_auc','brier_score']]

Model selection is based on validation PR-AUC. In the generated benchmark, logistic regression has slightly stronger validation performance than XGBoost, so the added nonlinear complexity is not selected.

In [ ]:
X_train, y_train = split_xy(frames['train'])
X_val, y_val = split_xy(frames['validation'])
model = build_candidate_models(X_train)['logistic_regression']
model.fit(X_train, y_train)
scores = predict_scores(model, X_val)
print('Validation ROC-AUC:', round(roc_auc_score(y_val, scores), 3))
print('Validation PR-AUC :', round(average_precision_score(y_val, scores), 3))

In [ ]:
selection = pd.read_json(ROOT / 'reports' / 'metrics' / 'model_selection.json', typ='series')
selection

The test set is not used in this notebook to choose a model or threshold. The locked test results are produced only after validation-based selection in the benchmark script.